Nikhil Sunder \
Student | ML Researcher | Open-Source Developer \
B.S.B.A. Quantitative Economics, Finance & Minor in Math \
University of Miami Herbert Business School \
[Github](https://github.com/nikhilxsunder) | [PyPI](https://pypi.org/user/nikhil.sunder/) | [Anaconda](https://anaconda.org/nikhil.sunder/) | [LinkedIn](https://www.linkedin.com/in/nikhil-sunder/) \
[ORCID](https://orcid.org/0009-0007-3323-1760) | [Zenodo](https://zenodo.org/search?q=metadata.creators.person_or_org.name:%22Sunder,+Nikhil%22) | [Handshake](https://miami.joinhandshake.com/profiles/6pcqp4) \
nss106@miami.edu \
(305) 409-3108

# NS-SDN (Non-Stationary Spectral Decompositon Network) For Econometric Applications

## Introduction

In my recent work and independent research I have come across an interesting paper from Stanford. In this paper the researchers created a neural network called the Sinusoidal Representation Network in which they make use of ${\sin{(x)}}$ for the activation function as opposed to soemthing more traditional such as the logistic sigmoid function or ${\tanh{(x)}}$. In using ${\sin{(x)}}$ for activation some advantages are observed. Sine waves allow representation of periodicity and cyclical behavior while still constraining values to the range ${(-1,1)}$. Current useful applications for representation of harmonic oscillators in neural network architecture include signal-processing and physics informed neural networks (PINNs). With this in mind, I seek to explore whether this type of neural network has applications in modeling econometric time series due to their composition of cyclicality and secular trend. This application also can be seen as an extension of the emerging econometric sub-field of econo-physics. I specifically seek to model implicit value shocks in macroeconomic time-series over discrete time under the assumption that in the relative short term, post shock they follow the pattern of a complex sinusoid.

## Design Concept

In this section I will give a rough overview of design and architecture as well as some basic model assumptions. The design portion will evaluate the network but not initialization and optimization as that will be part of the actual research portion itself. This network is simply a starting point for my idea and will be tweaked and re-evaluated through the buildout portion of my research prior to its application in forecasting.

### Network Overview

In pursuing this neural net as an econometric time series modeling tool, if proven advantageous it is reasonable to expect its use to be that of a replacement for older cyclical modeling methods such as Hodrick-Prescott filters, Kalman filters, LOESS decomposition, etc. Due to this fact we will follow the assumption that that our time series is some additive composition of a secular trend component, cyclical component, and an error term: 
$$
{Y_t = T_t + C_t + \epsilon(t)}
$$\

In expanding this definition to fit our sinusoidal model we can expand ${C_t}$:
$$
{C_t=A(t)\sin{(\omega(t))}}
$$

in which ${A(t)}$, and ${\omega(t)}$ represents our amplitude and phase functions respectively. 


In creating a general representation of the desired network design we have the following:
$$
{f(x)=B(x)+\sum_{k=1}^{K} A_k(x)\sin{(\theta_k(x))}}
$$
Where ${B,A_k,\theta_k}$ are all learned functions.
\
For each learned function we expand as follows:\
\
**Trend:**\
${B(x)= w_{B}^{\top} h+b_B}$
\
\
**Amplitudes:**\
${A(x)\in{\mathbb{R}^K}=softplus(W_Ah+b_A)}$
\
\
**Frequencies:**\
${\omega(x)\in{\mathbb{R}^K}=\omega_{base}+\Delta\omega(x)}$ in which ${\Delta\omega(x)=W_{\omega}h+b_{\omega}}$ 
\
\
Which when composed and expanded gives us:
$$
{f(x)=w_{B}^{\top}h(x)+b_B+\sum_{k=1}^{K} \sigma_{+}(w_{A,k}^{\top}h(x)+b_{A,k})\sin{(\theta_k(x))}}
$$
\
In which:
$${\sigma_{+}(x)=softplus(x)=log(1+e^x)=max(x,0)+log(1+e^{-|x|})}$$
$${h(x)=\phi(W_L \phi(...\phi(W_1x+b_1))=b_L)}$$
\
**Phase:**\
${\theta_k(x)}$

Now because we are operating under the assumption that our time series has time-dependent and patternistic micro-structures, the final progression to the above would be to express the network in terms of discrete time ${t_n}$. It is important to also note that below I will add a phase offset to account for amplitude and frequency distortion that would otherwise occur, due to the fact that we are modeling time-dependent patterns which we expect to be initiated by a shock. In addition to this, in order to create symmetry and number stability, we reparamatrize the activation into an additive composition of both sine and cosine. The final network is broken down as follows:

**Time Grid:**\
${t_0<t_1<...<t_N, \Delta t_n := t_n-t_{n-1}}$

**Latent State:**\
${h(t_n)=\phi(t_n)\in{\mathbb{R}^d}}$

**Trend Component:**\
${B(t_n)=w_{B}^{\top} h(t_n)+b_B}$

**Amplitude:**\
${A(t_n)= softplus(W_A h(t_n)+b_{A})\in{\mathbb{R}_{>0}^{K}}}$

Component-wise:\
${A_k(t_n)=softplus(w_{A,k}^{\top}h(t_n)+b_{A,k})}$

**Frequency** (Instantaneous Angular Frequency):\
${\omega(t_n)=\omega_{base}+W_{\omega}h(t_n)+b_{\omega}\in{\mathbb{R}^K}}$

Component-wise:\
${\omega_k(t_n)=\omega_{base,k}+w_{\omega,k}^{\top} h(t_n)+b_{\omega,k}}$

**Phase** (Cumulative Discrete-Time)\
$\phi_k(t_n) = \begin{cases} \theta_{k,0} &  n=0 \\ \theta_k{(t_{n-1})+ \omega_k{(t_n)} \Delta{t_n}} & n \geq{1} \end{cases}$

**Phase Offset** (Horizontal Shift)\
${\phi(t_n)=W_{\phi}h(t_n)+b_\phi \in{\mathbb{R}^K}}$
\
\
With the identity: ${\sin{(\theta+\phi)}=\sin{(\theta)}\cos{(\phi)}+\cos{(\theta)}\sin{(\phi)}}$ we have the network:

$${f(t_n)= B(t_n)+\sum_{k=1}^{K} A_k(t_n)[\sin(\theta_k(t_n))\cos(\phi_k(t_n))+\cos(\theta_k(t_n))\sin(\phi_k(t_n))]}$$

In the network above what we have is something very close to an FM-Synthesis Network or Neural Oscillator Model, but augmented with the "learned envelopes" from neural audio synthesis blocks and learned sinusoidal oscillators. Based on its design, properties, and closest relative, this network is best described as a "Non-Stationary Spectral Decompositon Network".

## Econometric Application

In testing the efficacy of the NS-SDN in forecasting macroeconomic time series it seems relevant to compare its performance to ARIMA, SVAR, and Single Layer Feed Forward ANNs. In doing so we will compare each models ability to forecast values as a function of discrete time for series which we believe to be inherently cyclical with some periodicity. 